# Notebook 12: Chatbot embeddings

**Goal:** turn every app recipe into a vector (embedding), so the chatbot can find recipes that match a question in plain English (RAG: *retrieval-augmented generation*).

**How the chatbot will work (next step, in the app):**
1. The question becomes a vector with the same model.
2. We take the recipes with the most similar vectors (cosine similarity), after simple filters (time, calories).
3. Recipes that fit the user's cycle phase move up.
4. The best 3 go to the LLM (Groq, `gpt-oss-20b`), which answers **only from them**.

**Agreed decisions (19 Nov):**
1. Same **12,375 recipes as the app** (not the RAW_recipes sample from the first prototype), so answers can use phase labels and nutrients and open in the app's recipe popup.
2. Text per recipe: name, meal type, ingredients, nutrients the recipe is rich in (≥ 20% of daily value), suitable phases, time, calories, then the description. The generic Food.com description (*"Make and share this … recipe from Food.com"*) is dropped.
3. Model: **`all-MiniLM-L6-v2`** (384 dimensions) through **fastembed** (ONNX, ~90 MB, no torch), so it also runs on Streamlit Cloud.
4. We save only the vectors (`float16`, about 9.5 MB) and the recipe ids. The text sent to the LLM (ingredients, steps) is built from `recipes_app.parquet` when a question is asked.

**Outputs (`../app/data/`):** `chat_embeddings.npy`, `chat_ids.npy`, `chat_config.json`.

In [1]:
%pip install -q fastembed


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: /Users/sarah/Desktop/Final Project/my-project/.venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import re
import time

import numpy as np
import pandas as pd

DATA_DIR = "../app/data"                     # path strings
RECIPES_FILE = DATA_DIR + "/recipes_app.parquet"
EMB_FILE = DATA_DIR + "/chat_embeddings.npy"
IDS_FILE = DATA_DIR + "/chat_ids.npy"
CONFIG_FILE = DATA_DIR + "/chat_config.json"

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
RICH_MIN = 20          # % of daily value to call a recipe "rich in" a nutrient
PHASES = ["menstrual", "follicular", "ovulatory", "luteal"]

## 1. Recipes

In [3]:
recipes = pd.read_parquet(RECIPES_FILE)
print(recipes.shape)
recipes[["RecipeId", "Name", "meal_type", "TotalTime_min", "Calories"]].head()

(12375, 36)


,RecipeId,Name,meal_type,TotalTime_min,Calories
0,55,Betty Crocker's Southwestern Guacamole Dip,Snacks & appetizers,125,415.9
1,56,Buttermilk Pie,Desserts & baking,80,395.9
2,85,Butter Madeira Cake,Desserts & baking,175,373.8
3,113,Carrot Ginger Biscuits,Desserts & baking,50,127.0
4,118,Abby's Pecan Apple Cake,Desserts & baking,90,357.2


## 2. Building the text for each recipe

**Order matters:** the model reads at most ~256 word pieces, so the most useful facts come first and the free-text description comes last (if anything is cut, it is the end of the description).

In [4]:
NUTRIENTS = {"pdv_iron_mg": "iron", "pdv_vitc_mg": "vitamin C", "pdv_magnesium_mg": "magnesium",
             "pdv_calcium_mg": "calcium", "pdv_zinc_mg": "zinc", "pdv_vitb6_mg": "vitamin B6",
             "pdv_folate_ug": "folate", "pdv_vitb12_ug": "vitamin B12", "pdv_fiber_g": "fiber",
             "pdv_protein_g": "protein", "pdv_potassium_mg": "potassium"}
MEAL_TEXT = {"Main": "Main dish", "Salads & sides": "Salad or side dish", "Desserts & baking": "Dessert or baking",
             "Breakfast": "Breakfast", "Snacks & appetizers": "Snack or appetizer",
             "Sauces & dressings": "Sauce or dressing", "Other": ""}
BOILERPLATE = re.compile(r"^\s*make and share this .* recipe from food\.com\.?\s*$", re.I)

is_boiler = recipes["description"].fillna("").str.match(BOILERPLATE)
print(f"generic Food.com descriptions dropped: {is_boiler.sum():,} of {len(recipes):,} ({is_boiler.mean():.0%})")


def join_words(items):
    items = list(items)
    return items[0] if len(items) == 1 else ", ".join(items[:-1]) + " and " + items[-1]


def search_text(r):
    parts = [f"{r['Name']}."]
    meal = MEAL_TEXT.get(r["meal_type"], "")
    cat = str(r["RecipeCategory"] or "")
    if cat == "nan" or cat.lower() in meal.lower():      # skip a category that repeats the meal type
        cat = ""
    kind = ", ".join(x for x in [meal, cat] if x)
    if kind:
        parts.append(kind + ".")
    parts.append("Ingredients: " + ", ".join(r["parts_norm"]) + ".")
    rich = [name for col, name in NUTRIENTS.items() if r[col] >= RICH_MIN]
    if rich:
        parts.append("Rich in " + join_words(rich) + ".")
    good = [p for p in PHASES if r[f"label_{p}"]]
    if good:
        parts.append(f"Good for the {join_words(good)} phase{'s' if len(good) > 1 else ''}.")
    parts.append(f"Ready in {int(r['TotalTime_min'])} minutes. {r['Calories']:.0f} calories per serving.")
    desc = str(r["description"] or "").strip()
    if desc and not BOILERPLATE.match(desc):
        parts.append(re.sub(r"\s+", " ", desc)[:400])
    return " ".join(parts)


recipes["search_text"] = recipes.apply(search_text, axis=1)
for t in recipes["search_text"].sample(3, random_state=1):
    print(t, "\n")
print("words per text:", recipes["search_text"].str.split().str.len().describe()[["mean", "50%", "max"]].round(0).to_dict())

generic Food.com descriptions dropped: 5,101 of 12,375 (41%)
Easy Summer Cream Cheese Frosting. Dessert or baking. Ingredients: cream cheese, powdered sugar, vanilla extract, cool whip. Ready in 10 minutes. 352 calories per serving. 

Pumpkin Zucchini Bread. Dessert or baking, Breads. Ingredients: egg, sugar, pumpkin puree, butter, vanilla extract, all purpose flour, baking soda, baking powder, salt, cinnamon, nutmeg, clove, zucchini, walnut. Ready in 65 minutes. 176 calories per serving. Pumpkin adds a twist of fall to the ever popular zucchini bread. Haven't tried this one yet but it looked like a winner so I thought I'd share it. 

Instant Pot Rice Pudding. Dessert or baking. Ingredients: arborio rice, water, salt, milk, sugar, egg, vanilla extract, raisin. Rich in vitamin B12. Ready in 38 minutes. 220 calories per serving. 

words per text: {'mean': 59.0, '50%': 56.0, 'max': 158.0}


## 3. Embeddings

The first run downloads the model (~90 MB) once. Embedding 12,375 texts takes a few minutes on a laptop CPU.

In [5]:
from fastembed import TextEmbedding

model = TextEmbedding(MODEL_NAME)

t0 = time.time()
emb = np.array(list(model.embed(recipes["search_text"].tolist(), batch_size=64)), dtype=np.float32)
emb /= np.linalg.norm(emb, axis=1, keepdims=True)          # unit length, so dot product = cosine similarity
print(emb.shape, f"in {time.time() - t0:.0f} s")

/Users/sarah/Desktop/Final Project/my-project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 5 files: 100%|██████████| 5/5 [00:30<00:00,  6.02s/it]


(12375, 384) in 86 s


In [6]:
np.save(EMB_FILE, emb.astype(np.float16))
np.save(IDS_FILE, recipes["RecipeId"].to_numpy(np.int64))
config = {"model": MODEL_NAME, "dim": int(emb.shape[1]), "n_recipes": int(len(recipes)),
          "rich_min_pdv": RICH_MIN, "built": time.strftime("%Y-%m-%d")}
with open(CONFIG_FILE, "w") as f:
    json.dump(config, f, indent=2)

import os
print(f"saved {EMB_FILE} ({os.path.getsize(EMB_FILE) / 1024**2:.1f} MB), {IDS_FILE}, {CONFIG_FILE}")

saved ../app/data/chat_embeddings.npy (9.1 MB), ../app/data/chat_ids.npy, ../app/data/chat_config.json


## 4. Search test

Same idea as the first prototype: simple limits from the question (*under 30 minutes*, *under 400 calories*), then cosine similarity. We use the saved `float16` file, exactly as the app will.

In [7]:
E = np.load(EMB_FILE).astype(np.float32)


def parse_constraints(text):
    text = text.lower()
    max_minutes = max_calories = None
    m = re.search(r"(?:under|less than|within|below|max)\s+(\d+)\s*(hours?|hrs?|minutes?|mins?)", text)
    if m:
        n = int(m.group(1))
        max_minutes = n * 60 if m.group(2).startswith("h") else n
    c = re.search(r"(?:under|less than|below|max)\s+(\d+)\s*(?:k?cal)", text)
    if c:
        max_calories = int(c.group(1))
    return max_minutes, max_calories


def search(question, k=3, phase=None):
    max_minutes, max_calories = parse_constraints(question)
    q = np.array(list(model.embed([question])), dtype=np.float32)[0]
    q /= np.linalg.norm(q)
    sim = E @ q
    ok = np.ones(len(recipes), bool)
    if max_minutes:
        ok &= recipes["TotalTime_min"].to_numpy() <= max_minutes
    if max_calories:
        ok &= recipes["Calories"].to_numpy() <= max_calories
    sim = np.where(ok, sim, -1)
    top = np.argsort(-sim)[:k]
    out = recipes.iloc[top][["Name", "meal_type", "TotalTime_min", "Calories"]].copy()
    out["similarity"] = sim[top].round(3)
    if phase:
        out["fits_phase"] = recipes.iloc[top][f"label_{phase}"].to_numpy()
    return out


questions = [
    "quick vegetarian dinner under 30 minutes",
    "something with iron for my period",
    "high protein breakfast",
    "low calorie chicken salad under 400 calories",
    "what can I cook with eggs, spinach and cheese?",
    "healthy chocolate dessert",
    "lentil soup",
    "snack rich in magnesium",
    "easy fish for two",
    "comfort food for a cold evening",
]
for q in questions:
    print(">>", q)
    print(search(q).to_string(index=False), "\n")

>> quick vegetarian dinner under 30 minutes
                          Name      meal_type  TotalTime_min  Calories  similarity
Stuffed and Barbecued Tomatoes Salads & sides             25     110.5       0.590
        Quick Shrimp and Pasta           Main             25     539.4       0.586
         Curried Tofu Scramble      Breakfast             15     219.7       0.580 

>> something with iron for my period
                          Name          meal_type  TotalTime_min  Calories  similarity
              Oregano dressing Sauces & dressings             10     251.2       0.405
Chocolate Covered Strawberries  Desserts & baking             25     249.4       0.393
            Walnut Parsley Dip Sauces & dressings              5     108.6       0.390 

>> high protein breakfast
                                  Name meal_type  TotalTime_min  Calories  similarity
Breakfast Egg Casserole - High Protein Breakfast             55     447.6       0.646
         Quick Healthy Tasty Breakfas

## 5. Phase check

The app will also move phase-suitable recipes up. Here we only look at how many of the plain top-10 already fit a phase, to see how much that step will matter.

In [8]:
for q, ph in [("something with iron for my period", "menstrual"), ("dinner for PMS cravings", "luteal"),
              ("fresh light lunch", "follicular")]:
    res = search(q, k=10, phase=ph)
    print(f"{q!r} ({ph}): {res['fits_phase'].sum()} of 10 fit the phase")

'something with iron for my period' (menstrual): 9 of 10 fit the phase
'dinner for PMS cravings' (luteal): 3 of 10 fit the phase
'fresh light lunch' (follicular): 0 of 10 fit the phase


## Next step

Add the **💬 Ask** tab to the app: load these files once, embed the question, filter + rank (similarity, then a bonus for the user's phase), send the best 3 recipes to Groq, and show them as recipe cards (♡ and View recipe) under the answer.